# Logistic Regression and Gradient Descent

In this workshop session you will practice implementing a spam email classifier using a logistic regression model and use gradient descent to optimise the parameters.

We will use the [`spambase` dataset](https://archive.ics.uci.edu/dataset/94/spambase) from the UC Irvine Machine Learning Repository. To save you some time, I have saved the dataset as a `.csv` file that you can find on ELE. Make sure the file `spambase.csv` is in the same directory as this Python notebook, and then you can execute the following code to load database. When successful, the output should be
```
(4601, 57)
(4601,)
```

In [2]:
import numpy as np

data = np.genfromtxt("spambase.csv", delimiter=",", skip_header=1)

X = data[:, :-1]   # features
y = data[:, -1].astype(int)    # labels
print(X.shape)
print(y.shape)

(4601, 57)
(4601,)


In this dataset, each data sample has 57 inputs and 1 output.

* The inputs are features extracted from the text of an email. For example, the first dimension of the input features is the frequency of the word 'make' in the email; the second dimension is the frequency of the word 'address' in the email. You can find a description of all these input dimensions on this web page: https://archive.ics.uci.edu/dataset/94/spambase.
  
* The output is either 0 (not spam) or 1 (spam).

Below I have included a function `spambase_features` that computes the features from a string, so that you can play with your classifier with email texts later on.

In [3]:
import re

WORDS = [
"make","address","all","3d","our","over","remove","internet","order","mail",
"receive","will","people","report","addresses","free","business","email","you",
"credit","your","font","000","money","hp","hpl","george","650","lab","labs",
"telnet","857","data","415","85","technology","1999","parts","pm","direct",
"cs","meeting","original","project","re","edu","table","conference"
]

CHARS = ['[','!','$','#',';','(']

def spambase_features(email_text):
    
    # ---- WORD FEATURES ----
    words = re.findall(r"[A-Za-z0-9]+", email_text.lower())
    total_words = len(words)
    
    word_counts = {w:0 for w in WORDS}
    
    for w in words:
        if w in word_counts:
            word_counts[w] += 1
    
    word_freq = []
    for w in WORDS:
        if total_words == 0:
            word_freq.append(0)
        else:
            word_freq.append(100 * word_counts[w] / total_words)
    
    # ---- CHARACTER FEATURES ----
    total_chars = len(email_text)
    
    char_freq = []
    for c in CHARS:
        if total_chars == 0:
            char_freq.append(0)
        else:
            char_freq.append(100 * email_text.count(c) / total_chars)
    
    # ---- CAPITAL RUN FEATURES ----
    runs = re.findall(r"[A-Z]+", email_text)
    
    if len(runs) == 0:
        capital_avg = 0
        capital_longest = 0
        capital_total = 0
    else:
        lengths = [len(r) for r in runs]
        capital_avg = np.mean(lengths)
        capital_longest = max(lengths)
        capital_total = sum(lengths)
    
    return np.array(
        word_freq +
        char_freq +
        [capital_avg, capital_longest, capital_total]
    )

test_email = """
FREE money now!!! Click here to receive your free credit report.
"""

print(spambase_features(test_email))

[ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          9.09090909  0.
  0.          9.09090909  0.         18.18181818  0.          0.
  0.          9.09090909  9.09090909  0.          0.          9.09090909
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          4.54545455  0.          0.          0.          0.
  2.5         4.          5.        ]


Be careful that the features of a data sample is represented as a length-57 **row vector**.

## Exercise 1: Implement the Softmax function

Let's start with implementing of the `softmax` function first. It accepts a vector $v$ of numbers and returns the vector
$$
[\frac{\exp{(v_1)}}{\sum_{i}\exp(v_i)}, \cdots, \frac{\exp{(v_n)}}{\sum_{i}\exp(v_i)}]
$$
where $n$ is the length of $v$.

In [6]:
def softmax(v):
   """
   Calculates the vector below from the input vector v.

   $$
   [\frac{\exp{(v_1)}}{\sum_{i}\exp(v_i)}, \cdots, \frac{\exp{(v_n)}}{\sum_{i}\exp(v_i)}]
   $$

   """
   # Calculate the sum of exp(v_i) for each component i in the input vector v
   exp_sum = 0
   for i in range(len(v)):
      exp_sum += np.exp(v[i])
   
   # Initialise an array to hold the output vector
   output_vector = np.zeros(shape=len(v), dtype=float)

   # Calculate exp(v_i)/exp_sum for each component i in the input vector v
   for i in range(len(v)):
      output_vector[i] = np.exp(v[i]) / exp_sum
   
   return output_vector

Let's also write a function that applies softmax to each row of a matrix. You can literally applies the `softmax` function you have just defined to each row, but a more efficient way is using `numpy` functions to directly work on a matrix. For example, the function `np.sum(X, axis=1, keepdims=True)` sums each row of a matrix.

In [11]:
def softmax_all_rows(z):
    # Initialise a matrix to hold the result
    output_matrix = np.zeros((len(z), len(z[0])))

    # Iterate through each row in the input matrix z
    for i in range(len(z)):
        output_matrix[i] = softmax(z[i])
    
    return output_matrix

You can run the following cell to check the correctness of your code. The results should be
```
[0.09003057 0.24472847 0.66524096]
[[0.33333333 0.33333333 0.33333333]
 [0.09003057 0.24472847 0.66524096]]
```

In [12]:
print(softmax(np.array([1, 2, 3])))
print(softmax_all_rows(np.array([[1,1,1], [4,5,6]])))

[0.09003057 0.24472847 0.66524096]
[[0.33333333 0.33333333 0.33333333]
 [0.09003057 0.24472847 0.66524096]]


## Exercise 2: Compute the gradients

Now let's define the logistic regression model. Given an N-by-n matrix $X$, each row of $X$ is a data sample, and each data sample has $n$ features. Given an n-by-m matrix $A$ and a 1-by-m matrix $B$. The function `probs` computes
the probabilities of each sample being non-spam or being spam.

In [16]:
def probs(X, A, B):
    return softmax_all_rows(X @ A + B)

print(probs(X, np.random.randn(57, 2) * 0.01, np.random.randn(1, 2) * 0.01))

[[7.70607088e-01 2.29392912e-01]
 [9.97713197e-01 2.28680259e-03]
 [9.99922905e-01 7.70952511e-05]
 ...
 [6.86842466e-01 3.13157534e-01]
 [6.15796418e-01 3.84203582e-01]
 [5.40190501e-01 4.59809499e-01]]


Now you need to compute the gradients of the loss function of logistic regression w.r.t. A and B. The function takes in an N-by-n matrix $X$ of data samples as described as above, as well as an N-by-2 matrix $Y$. Each row $Y_i$ is either $[1, 0]$ or $[0, 1]$, indicating that sample $i$ belongs to the first class or the second class.

In [ ]:
def grad(X, Y, A, B):
   # 

Now you need to implement gradient descent. The input $X$ and $Y$ are respectively the input and correct labels of the dataset in the format described above. Random initialisation of the parameters $A$ and $B$ has been done for you, so you just need to implement the updating for each step. 

In [ ]:
def train_softmax(X, Y, lr=0.01, steps=2000):
    N, n = X.shape
    m = Y.shape[1]
    
    # parameters
    A = np.random.randn(n, m) * 0.01
    B = np.random.randn(1, m) * 0.01

    for step in range(steps):
        # your code goes here

        if step % 200 == 0:
            P = probs(X, W, B)
            loss = -np.mean(np.sum(Y * np.log(P), axis=1))
            print(f"step {step} loss {loss:.4f}")

    return A, B

Now we can run the gradient descent algorithm on our dataset. A good practice in machine learning is to normalise each dimension of the input to have a mean of $0$ and standard deviation of $1$. This usually makes the process of learning faster.

In [ ]:
# normalise features
orig_mean = X.mean(axis=0)
orig_std = X.std(axis=0)

def transform_feature(X):
    return (X - orig_mean) / orig_std

X = transform_feature(X)

# one-hot labels
Y = np.zeros((X.shape[0], 2))
Y[np.arange(X.shape[0]), y] = 1

print(X)
print(Y)

Now we can feed the normalised input to our function `train_softmax`. You can play with different values of the learning rate to observe its impact.

In [ ]:
# train classifier
A, B = train_softmax(X_transformed, Y, lr=0.001, steps=2000)

With the learnt parameters $A$ and $B$, we can compute the probability of an email being spam by using `probs`.

In [ ]:
def spam_prob_email(text):
   return probs(transform_feature(spambase_features(text)), A, B)[0,1]

In [ ]:
# This should give a probability around 0.99
spam_prob_email("business opportunity make money now!!!")

In [ ]:
# This should give a probability around 0.007
spam_prob_email("workshop finished")

Finally, let's compute the accuracy of our trained model on our training set. For each data sample in the dataset, if our models belives the probability of it being spam is more than $0.5$, then we predict it's spam. Caculate the accurary of our predictions w.r.t. the truth $y$). The result should be something around 0.92 for the parameters `lr=0.001`, `steps=2000`.

In [ ]:
accuracy = # your code goes here
print("training accuracy:", accuracy)